In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn
import joblib

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.base import clone

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.3.3
Scikit-learn version: 1.7.2
NumPy version: 2.3.4


In [16]:
SEED = 3456

print("Seed:", SEED)

Seed: 3456


Load the Data

I load the training and test files. The training file has the target column `y`. The test file does not have the answers. I use the training data to build the model and the test data to make final predictions.

In [17]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

Train shape: (32940, 22)
Test shape: (8236, 20)

Train columns:
['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Test columns:
['id', 'age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [ ]:
# Explore the Data
display(train.head())

,id,age,job,marital,education,default,housing,loan,contact,month,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,0,39,management,single,basic.9y,unknown,yes,no,telephone,may,...,3,999,0,nonexistent,1.1,93.994,-36.4,4.856,5191.0,no
1,1,32,admin.,single,university.degree,no,yes,no,cellular,aug,...,2,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
2,2,41,technician,married,high.school,no,no,yes,cellular,aug,...,1,999,0,nonexistent,1.4,93.444,-36.1,4.963,5228.1,no
3,3,39,admin.,married,professional.course,no,yes,yes,telephone,may,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,4,51,blue-collar,married,basic.9y,unknown,no,no,telephone,jun,...,1,999,0,nonexistent,1.4,94.465,-41.8,4.864,5228.1,no


 Check Data Types and Missing Values

I check the data types and missing values. Some columns contain numbers, while others contain text. This information helps me choose the correct preprocessing steps.

In [19]:
train.info()

print("\nMissing values:")
print(train.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32940 entries, 0 to 32939
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              32940 non-null  int64  
 1   age             32940 non-null  int64  
 2   job             32940 non-null  object 
 3   marital         32940 non-null  object 
 4   education       32940 non-null  object 
 5   default         32940 non-null  object 
 6   housing         32940 non-null  object 
 7   loan            32940 non-null  object 
 8   contact         32940 non-null  object 
 9   month           32940 non-null  object 
 10  day_of_week     32940 non-null  object 
 11  duration        32940 non-null  int64  
 12  campaign        32940 non-null  int64  
 13  pdays           32940 non-null  int64  
 14  previous        32940 non-null  int64  
 15  poutcome        32940 non-null  object 
 16  emp.var.rate    32940 non-null  float64
 17  cons.price.idx  32940 non-null 

Check the Target Column

I count the `yes` and `no` answers. This shows if the target classes are balanced. If most clients say no, accuracy alone is not enough to evaluate the model.

In [20]:
print(train["y"].value_counts())

print("\nTarget percentages:")
print(train["y"].value_counts(normalize=True) * 100)

y
no     29229
yes     3711
Name: count, dtype: int64

Target percentages:
y
no     88.734062
yes    11.265938
Name: proportion, dtype: float64


Majority Class Baseline

I create a simple model that always predicts the most common class. It helps me understand why accuracy can be misleading for this dataset.

In [21]:
X_baseline = train.drop(columns=["y"])
y_baseline = train["y"]

majority_model = DummyClassifier(strategy="most_frequent")

majority_model.fit(X_baseline, y_baseline)

majority_predictions = majority_model.predict(X_baseline)

print(
    "Majority accuracy:",
    accuracy_score(y_baseline, majority_predictions)
)

Majority accuracy: 0.8873406193078324


Find Special Values

Some text values are `unknown`. They may mean that the information is not available. I count these values in each column before preprocessing.

In [22]:
for column in train.columns:
    if train[column].dtype == "object":
        unknown_count = (
            train[column]
            .astype(str)
            .str.lower()
            .eq("unknown")
            .sum()
        )

        if unknown_count > 0:
            print(column, ":", unknown_count)

job : 275
marital : 64
education : 1401
default : 6871
housing : 794
loan : 794
